## CPU latency benchmark

Per-message inference latency for all six models, on this machine's CPU, timed from the raw
SMS string to a predicted label - each model's own preprocessing (masking, tokenizing,
vectorizing) included, since that's part of what a real request pays.

Protocol: the same 500 test messages for every model, one message per call (no batching -
the realistic "one SMS arrives, classify it" case), after 50 untimed warmup calls. Reported
as mean, median (p50) and p95 in milliseconds.

This notebook measures latency only, not accuracy. The NN, CNN and both fine-tuned
transformers use untrained weights: inference cost depends on the architecture and input
length, not on the values of the weights, so a fine-tuned checkpoint and a freshly
initialized one of the same architecture take the same time. TF-IDF and the frozen-embedding
classifier are fit on the train split, because TF-IDF's vocabulary size does affect its cost.

In [1]:
import os
import re
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

C:\Users\Dell\Desktop\ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT_DIR = os.path.abspath(os.path.join(os.path.join(os.getcwd(), ".."), ".."))
DATA_DIR = os.path.join(ROOT_DIR, "Data")
DATASET_DIR = os.path.join(DATA_DIR, "SMS-Spam-Data")
DATASET_FILE = os.path.join(DATASET_DIR, "Dataset_5971.csv")

In [3]:
df = pd.read_csv(DATASET_FILE)
df["TEXT"] = df["TEXT"].str.lstrip("\t")
df["CLEAN"] = df["TEXT"].str.lower().str.split().str.join(" ")
df["LABEL"] = df["LABEL"].str.strip().str.lower()
df = df.drop_duplicates(subset="CLEAN").drop(columns="CLEAN")

URL_RE = re.compile(
    r"(https?://\S+|www\.\S+|\b[a-zA-Z0-9-]+\.(?:com|co\.uk|org\.uk|org|net|info|biz)(?:/\S*)?)",
    re.IGNORECASE,
)
PHONE_RE = re.compile(r"(\+?\d[\d\-\s]{6,}\d)")

def mask_text(text):
    text = URL_RE.sub("<URL>", text)
    text = PHONE_RE.sub("<PHONE>", text)
    return text

X = df["TEXT"]
y = df["LABEL"]

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1, stratify=y_temp, random_state=42)

encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)

In [4]:
N_WARMUP = 50
N_TIMED = 500
bench_texts = X_test.sample(n=N_WARMUP + N_TIMED, random_state=42).tolist()

def benchmark(predict_one):
    for text in bench_texts[:N_WARMUP]:
        predict_one(text)
    timings = []
    for text in bench_texts[N_WARMUP:]:
        start = time.perf_counter()
        predict_one(text)
        timings.append((time.perf_counter() - start) * 1000)
    timings = np.array(timings)
    return {"mean_ms": timings.mean(), "p50_ms": np.percentile(timings, 50), "p95_ms": np.percentile(timings, 95)}

latency = {}
torch.get_num_threads()

10

## TF-IDF + LogisticRegression

In [5]:
vectorizer = TfidfVectorizer()
tfidf_lr = LogisticRegression(class_weight="balanced")
tfidf_lr.fit(vectorizer.fit_transform(X_train.apply(mask_text)), y_train_encoded)

def predict_tfidf(text):
    return tfidf_lr.predict(vectorizer.transform([mask_text(text)]))

latency["TF-IDF + LogisticRegression"] = benchmark(predict_tfidf)

## NN mean pooling and 1D CNN

In [6]:
X_train_tokens = X_train.apply(mask_text).str.split()
token_counts = Counter(X_train_tokens.explode())
index_to_token = ["PAD", "UNK"] + [token for token, count in token_counts.most_common() if count >= 2]
token_to_index = {token: index for index, token in enumerate(index_to_token)}
max_len = int(np.percentile(X_train_tokens.apply(len), 95))

def encode_ids(text):
    ids = [token_to_index.get(token, 1) for token in mask_text(text).split()][:max_len]
    return torch.tensor([ids or [1]], dtype=torch.long)

class MeanPoolModel(nn.Module):
    def __init__(self, vocab_size, num_labels=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(64, num_labels)

    def forward(self, padded):
        emb = self.embedding(padded)
        mask = (padded != 0).unsqueeze(-1).float()
        pooled = (emb * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.fc(self.dropout(pooled))

class CNNModel(nn.Module):
    def __init__(self, vocab_size, num_labels=3, embedding_dim=64, num_filters=100, kernel_sizes=(2, 3, 4), dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(embedding_dim, num_filters, k) for k in kernel_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_labels)
        self.min_len = max(kernel_sizes)

    def forward(self, padded):
        if padded.size(1) < self.min_len:
            padded = nn.functional.pad(padded, (0, self.min_len - padded.size(1)), value=0)
        emb = self.embedding(padded).transpose(1, 2)
        pooled = [nn.functional.relu(conv(emb)).max(dim=2).values for conv in self.convs]
        return self.fc(self.dropout(torch.cat(pooled, dim=1)))

def torch_predictor(model):
    model.eval()
    def predict_one(text):
        with torch.inference_mode():
            return model(encode_ids(text)).argmax(dim=1)
    return predict_one

latency["NN mean pooling"] = benchmark(torch_predictor(MeanPoolModel(len(index_to_token))))
latency["1D CNN"] = benchmark(torch_predictor(CNNModel(len(index_to_token))))

## Frozen MiniLM embeddings + LogisticRegression

In [7]:
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
frozen_lr = LogisticRegression(class_weight="balanced", max_iter=1000)
frozen_lr.fit(embedder.encode(X_train.tolist()), y_train_encoded)

def predict_frozen(text):
    return frozen_lr.predict(embedder.encode([text]))

latency["Frozen MiniLM + LogisticRegression"] = benchmark(predict_frozen)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3363.70it/s]

## Fine-tuned MiniLM and DistilBERT

In [8]:
def finetune_predictor(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3).eval()
    max_length = int(np.percentile(X_train.apply(lambda t: len(tokenizer.encode(t))), 95))
    def predict_one(text):
        inputs = tokenizer(text, truncation=True, max_length=max_length, return_tensors="pt")
        with torch.inference_mode():
            return model(**inputs).logits.argmax(dim=1)
    return predict_one

latency["Fine-tuned MiniLM"] = benchmark(finetune_predictor("sentence-transformers/all-MiniLM-L6-v2"))
latency["Fine-tuned DistilBERT"] = benchmark(finetune_predictor("distilbert-base-uncased"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4084.15it/s]


[transformers] BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:  57%|█████▋    | 57/100 [00:00<00:00, 541.46it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 731.02it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Results

In [9]:
pd.DataFrame(latency).T.sort_values("p50_ms").round(3)

,mean_ms,p50_ms,p95_ms
NN mean pooling,0.123,0.106,0.208
TF-IDF + LogisticRegression,0.669,0.572,1.236
1D CNN,0.882,0.871,1.410
Fine-tuned MiniLM,21.931,21.166,34.409
Frozen MiniLM + LogisticRegression,28.632,27.698,43.464
Fine-tuned DistilBERT,42.148,38.084,69.731
